# Detección de Fraude — Redes Neuronales (MLP)

**Dataset:** Credit Card Fraud Detection (Kaggle, features anonimizadas vía PCA).
**Tarea:** clasificación binaria **extremadamente desbalanceada** (fraude ≈ 0.17 %).
**Métrica principal:** **PR-AUC** (Average Precision), *no* accuracy.

### Por qué este enfoque (mapa a la rúbrica)
| Rúbrica | Dónde se cubre en este notebook |
|---|---|
| 1. Preprocesamiento (escalado crítico para redes) | §2 — escalado robusto de `Amount`/`Time`, split estratificado, *sin fuga de datos* |
| 3. Redes + validación cruzada | §4–§5 (MLP 1–3 capas), §8 (StratifiedKFold) |
| 4. Optimización de pérdida / pesos | §3 — `pos_weight` en `BCEWithLogitsLoss` + `weight_decay` (L2) |
| 5. Interpretación y juicio | §6–§10 — PR-AUC, ajuste de umbral, AdaBoost vs MLP |

> **Idea clave:** con 0.17 % de positivos, un modelo que prediga "todo legítimo" logra ~99.83 % de accuracy y **0 % de recall**. Por eso medimos con PR-AUC y ajustamos el umbral, no con accuracy.

---
### Integración en el repo
Estas celdas son la **sección "Redes Neuronales"** del notebook `3_Modelos_de_Ensamble_y_Redes`. Pégalas debajo de las celdas de RF/AdaBoost/XGBoost de tu equipo.

⚠️ **Para que la comparativa final sea justa**, el split debe ser idéntico al de tus compañeros: usa la **misma `SEED`, mismas fracciones (60/20/20) y el mismo `stratify=y`** sobre el mismo `df`. `train_test_split` es determinista con `random_state` fijo, así que con esos tres acuerdos el `X_test` es el mismo para todos. *Acuérdenlo antes de entrenar.*


## 0. Reproducibilidad e imports
Fijamos todas las semillas (Python, NumPy, PyTorch) para que el resultado sea reproducible, como exige el entregable (`seed fijo`).

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    confusion_matrix, classification_report, f1_score,
)
from sklearn.ensemble import AdaBoostClassifier

SEED = 42

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)  # CPU es suficiente: la red es pequeña


## 1. Configuración y carga de datos
Cargamos con `kagglehub`, igual que en `1_Preprocesamiento_y_EDA.ipynb`, para no duplicar rutas locales.

In [ ]:
import kagglehub

# Hiperparámetros del MLP (centralizados para reproducibilidad)
HIDDEN_DIMS = [32, 16]   # 2 capas ocultas (la rúbrica pide 1–3)
DROPOUT     = 0.3
LR          = 1e-3
WEIGHT_DECAY= 1e-4       # = regularización L2 (conecta con el tema Ridge del proyecto)
BATCH_SIZE  = 2048
MAX_EPOCHS  = 40
PATIENCE    = 6          # early stopping sobre PR-AUC de validación

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))
print("Forma:", df.shape)
print(df["Class"].value_counts())
frac = df["Class"].mean()
print(f"Proporción de fraude: {frac:.4%}")


## 2. Preprocesamiento (crítico para redes neuronales)

**Razonamiento (para el informe):**
- Las variables `V1`–`V28` ya son **componentes principales (PCA)**, por lo que están centradas y en escalas comparables. *No* las re-escalamos para no destruir esa estructura.
- `Amount` y `Time` **no** pasaron por PCA y tienen rangos enormes y colas largas → una red neuronal con entradas sin escalar sufre gradientes mal condicionados. Usamos **`RobustScaler`** (mediana e IQR) porque es resistente a los *outliers* severos típicos del fraude.
- **Sin fuga de datos:** el escalador se ajusta **solo con el conjunto de entrenamiento** y luego se aplica a validación/test.
- **Split estratificado** para preservar la proporción de fraude (~0.17 %) en cada partición; con clases tan raras, un split aleatorio simple podría dejar casi sin positivos a validación/test.

In [ ]:
X = df.drop(columns=["Class"]).copy()
y = df["Class"].astype(int).values

# Split estratificado: 60% train / 20% val / 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED)
# 0.25 * 0.80 = 0.20 -> val = 20%

# Escalar SOLO Amount y Time; las V* ya vienen de PCA
cols_to_scale = ["Amount", "Time"]
scaler = RobustScaler().fit(X_train[cols_to_scale])

for part in (X_train, X_val, X_test):
    part[cols_to_scale] = scaler.transform(part[cols_to_scale])

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  fraude en {name}: {yy.mean():.4%}  ({yy.sum()} casos)")


## 3. Manejo del desbalance vía la función de pérdida (Rúbrica 4)

En lugar de re-muestrear (SMOTE/undersampling), penalizamos más fuerte los errores sobre la clase minoritaria **dentro de la función de pérdida**. Esto es exactamente "manejo de hiperparámetros de pérdida/pesos en clasificación".

Usamos `BCEWithLogitsLoss(pos_weight = N_neg / N_pos)`: cada fraseo mal clasificado pesa ~`N_neg/N_pos` veces más que un legítimo. Más abajo (§ extra) puedes barrer `pos_weight` y observar el *trade-off* precisión↔recall — un resultado empírico ideal para el informe.

In [ ]:
n_pos = int(y_train.sum())
n_neg = int(len(y_train) - n_pos)
pos_weight_value = n_neg / n_pos
print(f"pos_weight = {pos_weight_value:.1f}")

def make_loader(Xdf, yarr, shuffle):
    Xt = torch.tensor(Xdf.values, dtype=torch.float32)
    yt = torch.tensor(yarr, dtype=torch.float32).unsqueeze(1)
    ds = TensorDataset(Xt, yt)
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, generator=g)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

INPUT_DIM = X_train.shape[1]


## 4. Arquitectura del MLP

**Razonamiento:** red **pequeña a propósito**. Con tan pocos positivos, una red grande sobreajusta el ruido de la clase mayoritaria. Bloques: `Linear → BatchNorm → ReLU → Dropout`.
- **BatchNorm** estabiliza el entrenamiento con entradas heterogéneas.
- **Dropout (0.3)** regulariza.
- Salida = **1 logit** (sin sigmoide); la sigmoide vive dentro de `BCEWithLogitsLoss` por estabilidad numérica.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))  # logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

set_seed(SEED)
model = MLP(INPUT_DIM, HIDDEN_DIMS, DROPOUT).to(DEVICE)
print(model)
print("Parámetros:", sum(p.numel() for p in model.parameters()))


## 5. Entrenamiento con *early stopping* sobre PR-AUC

Validamos cada época con **Average Precision (PR-AUC)** y guardamos los pesos del mejor estado. `weight_decay` del optimizador Adam aporta **regularización L2**, el mismo principio Ridge que el proyecto enfatiza para trabajar con componentes principales.

In [ ]:
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs, ys = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        p = torch.sigmoid(model(xb)).cpu().numpy().ravel()
        probs.append(p); ys.append(yb.numpy().ravel())
    return np.concatenate(probs), np.concatenate(ys)

def train_model(model, train_loader, val_loader, max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=True):
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], device=DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history = {"train_loss": [], "val_loss": [], "val_ap": []}
    best_ap, best_state, bad = -1.0, None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)

        # validación
        model.eval()
        with torch.no_grad():
            vloss = 0.0
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vloss += criterion(model(xb), yb).item() * xb.size(0)
            vloss /= len(val_loader.dataset)
        vp, vy = predict_proba(model, val_loader)
        val_ap = average_precision_score(vy, vp)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(vloss)
        history["val_ap"].append(val_ap)
        if verbose:
            print(f"Época {epoch:02d} | train {train_loss:.4f} | val {vloss:.4f} | val PR-AUC {val_ap:.4f}")

        if val_ap > best_ap:
            best_ap, best_state, bad = val_ap, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                if verbose: print(f"Early stopping en época {epoch} (mejor PR-AUC val: {best_ap:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_ap

set_seed(SEED)
model = MLP(INPUT_DIM, HIDDEN_DIMS, DROPOUT).to(DEVICE)
model, history, best_val_ap = train_model(model, train_loader, val_loader)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Pérdida (BCE ponderada)"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(history["val_ap"], color="green")
ax[1].set_title("PR-AUC de validación"); ax[1].set_xlabel("época")
plt.tight_layout(); plt.show()


## 6. Evaluación en TEST — PR-AUC primero

Reportamos **PR-AUC** (métrica principal) y ROC-AUC como referencia. La curva PR muestra el *trade-off* real precisión↔recall para distintos umbrales.

In [ ]:
test_p, test_y = predict_proba(model, test_loader)

ap  = average_precision_score(test_y, test_p)
roc = roc_auc_score(test_y, test_p)
print(f"TEST  PR-AUC (Average Precision): {ap:.4f}")
print(f"TEST  ROC-AUC:                    {roc:.4f}")

prec, rec, thr = precision_recall_curve(test_y, test_p)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec, label=f"MLP (PR-AUC={ap:.3f})")
plt.axhline(test_y.mean(), ls="--", color="gray", label=f"baseline aleatorio={test_y.mean():.4f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Curva Precision-Recall (test)")
plt.legend(); plt.tight_layout(); plt.show()


## 7. Ajuste de umbral (decisión de negocio)

El modelo entrega una **probabilidad**; el umbral 0.5 por defecto raramente es óptimo en datos desbalanceados. Elegimos el umbral que **maximiza F1** sobre la curva PR. En un caso real, este umbral se fija según el costo de un fraude no detectado vs. el costo de revisar un falso positivo — vale la pena discutirlo en el informe.

In [ ]:
f1s = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = int(np.nanargmax(f1s[:-1]))   # thr tiene len-1 vs prec/rec
best_thr = thr[best_idx]
print(f"Umbral óptimo (max F1): {best_thr:.4f}  ->  F1={f1s[best_idx]:.4f}, "
      f"precision={prec[best_idx]:.4f}, recall={rec[best_idx]:.4f}")

y_pred = (test_p >= best_thr).astype(int)
cm = confusion_matrix(test_y, y_pred)
print("\nMatriz de confusión [filas=real, cols=pred]:\n", cm)
print("\n", classification_report(test_y, y_pred, digits=4, target_names=["legítima", "fraude"]))


## 8. Validación cruzada (Rúbrica 3 — evitar sobreajuste)

`StratifiedKFold` re-entrena el MLP en *k* particiones y reporta **PR-AUC media ± desviación**, evidencia de que el resultado no depende de un único split afortunado. Se re-ajusta el escalador *dentro de cada fold* (sin fuga). Reducimos épocas para que corra en CPU en tiempo razonable.

In [ ]:
def cv_mlp(X_all, y_all, n_splits=5, max_epochs=15):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    aps = []
    Xv = X_all.values if hasattr(X_all, "values") else X_all
    for fold, (tr, va) in enumerate(skf.split(Xv, y_all), 1):
        Xtr = X_all.iloc[tr].copy(); Xva = X_all.iloc[va].copy()
        ytr, yva = y_all[tr], y_all[va]
        sc = RobustScaler().fit(Xtr[cols_to_scale])
        Xtr[cols_to_scale] = sc.transform(Xtr[cols_to_scale])
        Xva[cols_to_scale] = sc.transform(Xva[cols_to_scale])

        global pos_weight_value
        pw_old = pos_weight_value
        pos_weight_value = (ytr == 0).sum() / (ytr == 1).sum()

        tl = make_loader(Xtr, ytr, shuffle=True)
        vl = make_loader(Xva, yva, shuffle=False)
        set_seed(SEED + fold)
        m = MLP(Xtr.shape[1], HIDDEN_DIMS, DROPOUT).to(DEVICE)
        m, _, _ = train_model(m, tl, vl, max_epochs=max_epochs, patience=4, verbose=False)
        p, yy = predict_proba(m, vl)
        ap_fold = average_precision_score(yy, p)
        aps.append(ap_fold)
        pos_weight_value = pw_old
        print(f"  Fold {fold}: PR-AUC = {ap_fold:.4f}")
    aps = np.array(aps)
    print(f"\nPR-AUC CV: {aps.mean():.4f} ± {aps.std():.4f}")
    return aps

# Usa train+val (deja test intacto). Comenta si quieres ahorrar tiempo.
_ = cv_mlp(X_trainval.reset_index(drop=True), y_trainval, n_splits=5, max_epochs=15)


## 9. Comparación: AdaBoost vs MLP

La consigna pide contrastar **AdaBoost frente a redes neuronales** con PR-AUC. Entrenamos un AdaBoost con la misma partición y superponemos sus curvas PR.

In [ ]:
set_seed(SEED)
ada = AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=SEED)
ada.fit(X_train.values, y_train)            # árboles: no requieren escalado
ada_p = ada.predict_proba(X_test.values)[:, 1]
ada_ap = average_precision_score(test_y, ada_p)
ada_roc = roc_auc_score(test_y, ada_p)
print(f"AdaBoost  PR-AUC={ada_ap:.4f} | ROC-AUC={ada_roc:.4f}")
print(f"MLP       PR-AUC={ap:.4f} | ROC-AUC={roc:.4f}")

ap_prec, ap_rec, _ = precision_recall_curve(test_y, ada_p)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec, label=f"MLP (PR-AUC={ap:.3f})")
plt.plot(ap_rec, ap_prec, label=f"AdaBoost (PR-AUC={ada_ap:.3f})")
plt.axhline(test_y.mean(), ls="--", color="gray", label="baseline aleatorio")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("MLP vs AdaBoost (test)")
plt.legend(); plt.tight_layout(); plt.show()


## 10. Interpretación para el informe (Rúbrica 5)

Plantillas de razonamiento — sustituye con tus números reales:

- **Métrica correcta:** con 0.17 % de fraude, accuracy es engañosa; PR-AUC mide directamente la capacidad de ordenar fraudes por encima de legítimas. El baseline aleatorio de la curva PR es ≈ la prevalencia (~0.0017).
- **MLP vs AdaBoost:** comenta *quién ganó en PR-AUC y por qué*. Las redes capturan interacciones no lineales entre componentes PCA; AdaBoost concentra esfuerzo en los casos difíciles (boosting). Si quedan parejos, es señal de que la señal en estos datos es mayormente capturable por modelos relativamente simples bien regularizados.
- **Pesos en la pérdida:** sin `pos_weight`, la red colapsa a predecir "legítima" (recall ≈ 0). El `pos_weight` recupera recall a costa de precisión → el ajuste de umbral (§7) recoloca ese balance según el costo de negocio.
- **Regularización L2:** `weight_decay` (Ridge) es coherente con que las entradas son componentes principales correlacionados; estabiliza los pesos y reduce varianza.

> **Extra recomendado:** barre `pos_weight` ∈ {1, 5, 25, 100, pos_weight_value} y grafica precisión y recall vs. el peso. Es la demostración empírica que pide la Rúbrica 4.

In [ ]:
# Guardado de artefactos reproducibles
import json as _json
os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/mlp_fraud.pt")
metrics = {
    "mlp_pr_auc": float(ap), "mlp_roc_auc": float(roc),
    "adaboost_pr_auc": float(ada_ap), "adaboost_roc_auc": float(ada_roc),
    "best_threshold": float(best_thr), "best_val_pr_auc": float(best_val_ap),
    "pos_weight": float(pos_weight_value), "seed": SEED,
}
with open("artifacts/mlp_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print("Guardado en artifacts/:", metrics)
